# HBase・Sparkの更新と利用再開

共有Hadoop配布物の更新後、停止済みのHBaseとSparkを更新する。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

# 配布物の切替

同じホストのMaster・RegionServerやクライアントも配布物を共有する。利用者の処理とサービスが停止している間に切り替える。

In [ ]:
!ansible-playbook -CDv {playbook_dir}/switch_upgrade_package.yml --tags hbase,spark -l {target_group}

HBase/Spark対象ホストのcurrentだけが変更されることを確認する。

In [ ]:
!ansible-playbook {playbook_dir}/switch_upgrade_package.yml --tags hbase,spark -l {target_group}

In [ ]:
!ansible hadoop_hbase_master -a "hbase version" -l {target_group}

In [ ]:
!ansible hadoop_spark -a "spark-submit --version" -l {target_group}

# Hadoopライブラリの確認

HBaseが使用するHadoopライブラリをクラスタの版にそろえる。同梱のjarを退避し、HBaseの起動スクリプトが取得するHadoopのclasspathを使用する。

In [ ]:
hbase_home = '/opt/hbase/current'
hbase_home


In [ ]:
!ansible 'hadoop_hbase_master:hadoop_hbase_regionserver' -m shell -a 'ls -l {hbase_home}/lib/hadoop-*.jar' -l {target_group}

In [ ]:
!ansible 'hadoop_hbase_master:hadoop_hbase_regionserver' --become -m file -a 'path={hbase_home}/lib-hadoop-backup state=directory mode=0755' -l {target_group}

In [ ]:
!ansible 'hadoop_hbase_master:hadoop_hbase_regionserver' --become -m shell -a 'mv -v {hbase_home}/lib/hadoop-*.jar {hbase_home}/lib-hadoop-backup/' -l {target_group}

In [ ]:
!ansible 'hadoop_hbase_master:hadoop_hbase_regionserver' -a 'hbase --internal-classpath classpath' -l {target_group}

In [ ]:
!ansible 'hadoop_hbase_master:hadoop_hbase_regionserver' -a 'hbase --internal-classpath org.apache.hadoop.util.VersionInfo' -l {target_group}

出力に更新先のHadoopが含まれ、退避したjarが含まれないことを確認する。

# HBaseの起動とデータ確認

In [ ]:
!ansible-playbook {playbook_dir}/start_hbase_master.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/start_hbase_regionserver.yml -l {target_group}

In [ ]:
hbase_check_table = 'upgrade_check'
hbase_check_table

HBaseの管理操作はhbaseユーザーで行う。Kerberos構成では、サーバーの認証設定を使ってZooKeeperにも接続する。

In [ ]:
if kerberos_enabled:
    !ansible hadoop_hbase_master --become --become-user hbase -m shell -a 'kinit -kt /etc/hbase/conf/hbase.keytab hbase/$(hostname -f)' -l {target_group}

In [ ]:
hbase_query = f"""status 'detailed'
get '{hbase_check_table}', 'before'
put '{hbase_check_table}', 'after', 'data:value', 'writable'
get '{hbase_check_table}', 'after'
"""
with open(os.path.join(work_dir, "hbase-after.rb"), "w") as query_file:
    query_file.write(hbase_query)
print(hbase_query)

In [ ]:
!ansible hadoop_hbase_master --become -m copy -a "src={work_dir}/hbase-after.rb dest=/tmp/hbase-upgrade-after.rb owner=hbase group=hbase mode=0644" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible hadoop_hbase_master --become --become-user hbase -a "hbase --auth-as-server shell -n /tmp/hbase-upgrade-after.rb" -l {target_group}
else:
    !ansible hadoop_hbase_master --become --become-user hbase -a "hbase shell -n /tmp/hbase-upgrade-after.rb" -l {target_group}

MasterとRegionServerが揃い、更新前の値と新しく書き込んだ値が取得できることを確認する。

# SparkのHadoopクライアント更新

Sparkが使用するHadoopクライアントをクラスタの版にそろえる。apiとruntimeを同じ版の組で入れ替える。

In [ ]:
hadoop_home = '/opt/hadoop/current'
spark_home = '/opt/spark/current'
(hadoop_home, spark_home)


In [ ]:
!ansible 'hadoop_spark:hadoop_spark_history' -m shell -a 'ls -l {spark_home}/jars/hadoop-client-api-*.jar {spark_home}/jars/hadoop-client-runtime-*.jar {hadoop_home}/share/hadoop/client/hadoop-client-api-*.jar {hadoop_home}/share/hadoop/client/hadoop-client-runtime-*.jar' -l {target_group}

In [ ]:
!ansible 'hadoop_spark:hadoop_spark_history' --become -m file -a 'path={spark_home}/jars-hadoop-backup state=directory mode=0755' -l {target_group}

In [ ]:
!ansible 'hadoop_spark:hadoop_spark_history' --become -m shell -a 'mv -v {spark_home}/jars/hadoop-client-api-*.jar {spark_home}/jars/hadoop-client-runtime-*.jar {spark_home}/jars-hadoop-backup/' -l {target_group}

In [ ]:
!ansible 'hadoop_spark:hadoop_spark_history' --become -m shell -a 'cp -v {hadoop_home}/share/hadoop/client/hadoop-client-api-*.jar {hadoop_home}/share/hadoop/client/hadoop-client-runtime-*.jar {spark_home}/jars/' -l {target_group}

In [ ]:
!ansible 'hadoop_spark:hadoop_spark_history' -a '{spark_home}/bin/spark-class org.apache.hadoop.util.VersionInfo' -l {target_group}

Sparkから読み込まれるHadoopが更新先の版であることを確認する。

# 履歴サービスの再開

In [ ]:
!ansible-playbook {playbook_dir}/start_mapreduce_historyserver.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/start_timelineservice.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/start_spark_historyserver.yml -l {target_group}

# YARNの利用確認

サービスが揃ったことを確認し、利用者アカウントでジョブを投入する。

In [ ]:
if kerberos_enabled:
    !ansible hadoop_client -m shell -a 'kinit -kt "$HOME/.hadoop.keytab" "$(id -un)"' -l {target_group}

In [ ]:
!ansible hadoop_client -a "yarn node -list -all" -l {target_group}

In [ ]:
!ansible hadoop_client -m shell -a "yarn jar /opt/hadoop/current/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar pi 2 1000" -l {target_group}

全NodeManagerがRUNNINGであり、πの計算結果が得られることを確認する。

[T12d](T12d_Hadoop%20-%20Simple%20Spark%20script%20for%20Test.ipynb)でSparkジョブを実行する。利用者の処理結果と履歴を確認してから利用を再開し、[更新の確認と確定](O12j_Hadoop%20-%20Verify%20and%20finalize%20upgrade.ipynb)へ進む。